In [1]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import json
import time
import re

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def clean_text(text):
    return ' '.join(text.split())

def process_character_systematically(character_name):
    base_url = "https://vi.wikipedia.org/wiki/"
    main_url = base_url + character_name.replace(" ", "_")
    
    print(f"--- Đang xử lý hệ thống cho: {character_name} ---")
    
    try:
        response = requests.get(main_url, headers=HEADERS, timeout=10)
        if response.status_code != 200: return None
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 1. Quét mục lục (TOC) để lấy các link sự kiện/mục nhỏ
        sections = []
        # Lấy phần "Đầu" (Giới thiệu)
        sections.append({"title": "Giới thiệu", "url": main_url})
        
        toc_links = soup.select('.vector-toc-link')
        for link in toc_links:
            href = link.get('href')
            if href and href.startswith('#'):
                title = link.find(class_='vector-toc-text').get_text(strip=True)
                # Loại bỏ số thứ tự ở đầu tiêu đề (ví dụ "1 Tên gọi" -> "Tên gọi")
                title = re.sub(r'^\d+\s+', '', title)
                sections.append({
                    "title": title,
                    "url": main_url + href
                })

        # 2. Tải HTML chi tiết cho từng mục và lưu vào danh sách
        systematic_data = {
            "nhan_vat": character_name,
            "url_chinh": main_url,
            "chi_tiet": []
        }
        
        excel_rows = []

        for sec in sections:
            print(f"   + Đang lấy mục: {sec['title']}")
            res = requests.get(sec['url'], headers=HEADERS)
            if res.status_code == 200:
                # Lưu vào danh sách JSON
                systematic_data["chi_tiet"].append({
                    "muc": sec['title'],
                    "url_muc": sec['url'],
                    "html": res.text
                })
                # Chuẩn bị dòng cho Excel
                excel_rows.append([character_name, sec['title'], sec['url'], res.text])
            time.sleep(0.5)

        return systematic_data, excel_rows

    except Exception as e:
        print(f"Lỗi: {e}")
        return None, []

# --- Chạy chương trình ---
char_name = "Đinh Tiên Hoàng"
json_output, excel_data = process_character_systematically(char_name)

if json_output:
    # 1. Xuất ra Excel để lưu trữ (Dễ quản lý theo hàng)
    df = pd.DataFrame(excel_data, columns=['Nhân vật', 'Tiêu đề mục', 'URL mục', 'Nội dung HTML'])
    df.to_excel(f"{char_name}_database.xlsx", index=False)
    
    # 2. Xuất ra JSON để hệ thống hóa (Dùng cho lập trình/AI)
    with open(f"{char_name}_system.json", 'w', encoding='utf-8') as f:
        json.dump(json_output, f, ensure_ascii=False, indent=4)

    print(f"\n==> THÀNH CÔNG: Đã tạo file Excel và JSON hệ thống cho {char_name}")

--- Đang xử lý hệ thống cho: Đinh Tiên Hoàng ---
   + Đang lấy mục: Giới thiệu
   + Đang lấy mục: Đầu
   + Đang lấy mục: 1Tên gọi
   + Đang lấy mục: 2Tuổi thơ
   + Đang lấy mục: 3Thống nhất đất nước
   + Đang lấy mục: 3.1Không phục Hậu Ngô Vương
   + Đang lấy mục: 3.2Loạn 12 sứ quân
   + Đang lấy mục: 3.3Dẹp các sứ quân
   + Đang lấy mục: 3.4Chiêu hàng
   + Đang lấy mục: 4Cai trị
   + Đang lấy mục: 4.1Mở nước Đại Cồ Việt
   + Đang lấy mục: 4.2Đóng đô Hoa Lư
   + Đang lấy mục: 4.3Xưng Hoàng Đế
   + Đang lấy mục: 4.4Ngoại giao
   + Đang lấy mục: 5Cái chết
   + Đang lấy mục: 5.1Nghi án cung đình
   + Đang lấy mục: 5.2Lời bàn
   + Đang lấy mục: 6Nhận định
   + Đang lấy mục: 7Tôn vinh - Di sản
   + Đang lấy mục: 8Gia đình
   + Đang lấy mục: 9Tham khảo
   + Đang lấy mục: 10Chú thích
   + Đang lấy mục: 11Xem thêm

==> THÀNH CÔNG: Đã tạo file Excel và JSON hệ thống cho Đinh Tiên Hoàng
